In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
%pip install geopandas
dbutils.library.restartPython()

In [0]:
from utils.observability import (
    create_control_table,
    get_last_processed_timestamp,
    insert_control_record,
)
from utils.shared import (
    add_timestamp,
    filter_uningested_data,
    load_table,
    notebook_exit,
)
from utils.silver import (
    add_columns_for_idempotency,
    add_scd_logic,
    add_weather_zone_coordinates,
    extract_col_from_json_string,
    identify_rows_to_expire,
    insert_new_rows,
    join_current_and_incoming_data,
    load_current_data,
    select_station_columns,
    update_expired_rows,
    validate_columns,
    validate_and_quarantine_rows,
    standardize_towns,
    convert_lat_lon_type,
)
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, max
from schema.silver import stations_table_schema
from logger.custom_logging import get_job_logger, set_up_logger
import logging
import geopandas as gpd

try:
    context = REPLACED_WITH_SPARK_CONF
    run_id = context.tags().get("runId").get()
except:
    # Fallback for when running as a file (not a notebook)
    run_id = "unknown"

log_info = {
    "layer": "silver",
    "job": "process_station_data",
    "dataset": "bronze_dev.electrovolt.ev_stations",
}

logger = set_up_logger()

log = get_job_logger(logger, **log_info, run_id=run_id)

log(logging.INFO, "Starting process_station_data")

# The variables below are for the control table which is used to track the last processed timestamp and observe run metrics

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
pipeline = "project_voltstream"
layer = f"{log_info["layer"]}.{log_info["job"]}"
error_message = None
records_processed = 0
records_failed = 0
last_processed_timestamp = None

try:
    control_table = create_control_table(spark)
    max_time = get_last_processed_timestamp(spark, control_table, pipeline, layer)
    df = spark.table("bronze_dev.electrovolt.ev_stations")
    df = filter_uningested_data(df, max_time, **log_info)
    df = extract_col_from_json_string(df, "raw_text")
    df = validate_columns(df, **log_info)
    df_valid, df_quarantine = validate_and_quarantine_rows(
        df,
        ["station_id", "latitude", "longitude", "status_id", "date_last_status_update"],
        **log_info,
    )
    records_processed = df_valid.count()
    records_failed = df_quarantine.count()
    df_view = df_valid.write.saveAsTable(
        "silver_dev.electrovolt.valid_stations", mode="overwrite"
    )
    df = select_station_columns(df_valid)
    df = standardize_towns(spark, df)
    df = convert_lat_lon_type(df)
    df = add_weather_zone_coordinates(df)
    df = add_timestamp(df)

    stations_df = spark.createDataFrame([], schema=stations_table_schema)
    stations_table = load_table(
        spark, "silver_dev.electrovolt.ev_stations", stations_df
    )

    table_df = spark.table("silver_dev.electrovolt.ev_stations")
    current_stations_dim = load_current_data(table_df, **log_info)

    joint_stations_df = join_current_and_incoming_data(
        df, current_stations_dim, "station_id"
    )

    joint_stations_df = add_scd_logic(
        joint_stations_df, "station_sk", ["status", "status_id"]
    )

    inserts_stations_df = add_columns_for_idempotency(
        joint_stations_df,
        ["station_id", "stg.date_last_status_update"],
        "station_sk",
        [
            "station_id",
            "title",
            "address",
            "town",
            "state_or_province",
            "latitude",
            "longitude",
            "weather_zone_lat",
            "weather_zone_lon",
            "status",
            "status_id",
            "date_last_status_update",
            "date_created",
            "ingest_timestamp",
        ],
    )

    expired_stations_df = identify_rows_to_expire(joint_stations_df, "station_sk")
    updated_stations_df = update_expired_rows(
        stations_table, expired_stations_df, "station_sk", **log_info
    )

    final_stations_df = insert_new_rows(
        stations_table, inserts_stations_df, "station_sk", **log_info
    )
    status = "success"
    last_processed_timestamp = df.agg(max("ingest_timestamp")).collect()[0][0]
    log(logging.INFO, "Finished process_station_data")


except Exception as e:
    status = "failed"
    error_message = str(e)
    raise

end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
insert_control_record(
    spark,
    control_table,
    pipeline,
    layer,
    last_processed_timestamp,
    records_processed,
    records_failed,
    start_time,
    end_time,
    error_message,
    status,
)
notebook_exit("SUCCESS")